In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BIO_LIP_COLUMNS = [
"PDB ID",
"Receptor chain",
"Resolution. '-1.00' stands for lack of resolution information, e.g. for NMR",
"Binding site number code",
"Ligand_ID",
"Ligand_chain",
"Ligand serial number",
"    Binding site residues (with PDB residue numbering)",
"    Binding site residues (with residue re-numbered starting from 1)",
"Catalytic site residues (different sites are separated by ';') (with PDB residue numbering)",
"    Catalytic site residues (different sites are separated by ';') (with residue re-numbered starting from 1)",
"EC number",
"GO terms",
"Binding affinity by manual survey of the original literature. The information in '()' is the PubMed ID",
"Binding affinity provided by the Binding MOAD database. The information in '()' is the ligand information in Binding MOAD",
"Binding affinity provided by the PDBbind-CN database. The information in '()' is the ligand information in PDBbind-CN",
"Binding affinity provided by the BindingDB database",
"UniProt ID",
"PubMed ID",
"Residue sequence number of the ligand (field _atom_site.auth_seq_id in PDBx/mmCIF format)",
"Receptor sequence"]


In [ ]:
df = pd.read_csv('/homes/hagayr/study/thesis/BioLiP_nr.txt', sep="\t", header=None, names=BIO_LIP_COLUMNS)


## Filter relevant columns

In [ ]:
relevant_columns = ["PDB ID", "Receptor chain", "Ligand_ID", "Ligand_chain"]
df = df[relevant_columns].astype('str')
df.shape


## Filter ligands with less than 2 proteins

In [ ]:
# Count occurrences of values in the specified column
value_counts = df["Ligand_ID"].value_counts()

# Get the values that occur more than once
values_to_keep = value_counts[value_counts > 1].index

# Filter the DataFrame to keep rows where the value in the specified column occurs more than once
df = df[df["Ligand_ID"].isin(values_to_keep)]
df = df[df["Ligand_chain"]=='A']
df.shape


## Leave unique protein per ligand

In [ ]:
filtered_df = df.groupby(["Ligand_ID", "PDB ID"]).apply(lambda x: x.loc[x['Receptor chain'].idxmin()])
filtered_df = df.groupby(["Ligand_ID", "PDB ID"]).apply(lambda x: x.loc[x['Ligand_chain'].idxmin()])
filtered_df.shape


# plot ligand histogram

In [ ]:
column_to_plot = "Ligand_ID"
value_counts = df[column_to_plot].value_counts()

# Plot the histogram
plt.bar(value_counts.index, value_counts.values, color='blue')

# Rotate x-axis labels for better readability if needed
plt.xticks(rotation=90)

# Add labels and title
plt.xlabel(column_to_plot)
plt.ylabel('Frequency')
plt.title('Histogram of {}'.format(column_to_plot))

# Show plot
plt.show()

In [ ]:
import os
import itertools
import json
import random

config = {
    "protein_names_list": "alligned_structures/ATP/atp.txt",
    "ligand_name": "ATP",
    "pdb_ligand_id": "H_ATP",
    "alligners": [
        {"name": "RANSACAlligner", "args": {}}
    ]
}
with open ('/homes/hagayr/study/thesis/allignment_playgroud/good_ligands.txt') as good_ligands_file:
        ligands =  [line.strip() for line in good_ligands_file]

# for id in df["Ligand_ID"].unique()[:500]:
for id in ligands:
    # Filter the DataFrame to include only rows with the current ID value
    filtered_rows = df[df['Ligand_ID'] == id]
    
    # Define the file path for the current ID
    ligand_repo = os.path.join("/homes/hagayr/study/thesis/allignment_playgroud/alligned_structures", id)
    os.makedirs(ligand_repo, exist_ok=True)
    config['protein_names_list'] = f'alligned_structures/{id}/{id}.txt'
    config['ligand_name'] = id
    config['pdb_ligand_id'] = f"H_{id}"

    with open(os.path.join(ligand_repo, "config.json"), "w") as config_file:
        json.dump(config, config_file, indent=4)

    
    # Write the filtered rows to a separate file
    num_of_pairs = 1
    unique_pairs = set()
    pairs_file = os.path.join(ligand_repo, f"{id}.txt")
    with open(pairs_file, "w") as output_file:
        combinations = list(itertools.combinations(filtered_rows.iterrows(), 2))
        random.shuffle(combinations)
        for row1, row2 in combinations:
            row1 = row1[1]
            row2 = row2[1]
            if row1['PDB ID'] != row2['PDB ID'] and (row1[0], row2[0]) not in  unique_pairs and (row2[0], row1[0]) not in  unique_pairs:
                unique_pairs.add((row1[0], row2[0]))
                num_of_pairs +=1
                output_file.write(f"{row1['PDB ID']}:{row1['Receptor chain']} {row2['PDB ID']}:{row2['Receptor chain']}\n")
                if num_of_pairs > 20:
                    break